In [ ]:
# Databricks notebook source
# tutorial_name: 02-Day7-DLT-Guide-UI-and-Troubleshooting
# notebookName: 02-Day7-DLT-Guide-UI-and-Troubleshooting
# COMMAND ----------

# DBTITLE 1,Paths
notepoint_rel = "hands-on/day-07/notebooks/02-Day7-DLT-Guide-UI-and-Troubleshooting.ipynb"
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # Change to your assigned u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
DAY7_PREFIX = f"{BASE_PATH}day07-{STUDENT_ID}"
# COMMAND ----------

# DBTITLE 1,Prerequisite check (interactive cluster — not a DLT pipeline)
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("OK: Day 5 Delta at P_BASIC is readable")
except Exception as e:
    print("P_BASIC read failed:", e)
    print("Complete Day 5 hands-on first.")

print("Suggested target schema naming: main.day07_" + STUDENT_ID.replace("-", "_"))
print("DAY7_PREFIX:", DAY7_PREFIX)
# COMMAND ----------


## Day 7 — Delta Live Tables (DLT): guide, UI, and troubleshooting

**Syllabus item 20.** Run this notebook on a **normal interactive cluster** (it does **not** use `import dlt`). Confirm paths here, then create a **DLT pipeline** whose libraries are either notebooks **03 → 04 → 05** **or** `pipelines/dlt_bronze_flights.py`, `pipelines/dlt_silver_flights.py`, `pipelines/dlt_gold_flights.py`

**What DLT is:** You declare **what** tables and views should contain (and **data quality rules**), not a manual sequence of notebook cells. Databricks resolves dependencies, can refresh incrementally, and shows **lineage** and **expectation** results in the pipeline UI.

**Medallion in this lab:** **Bronze** = ingest aligned to the pipeline (here: read existing Day 5 Delta `P_BASIC`). **Silver** = cleaned / filtered **view**. **Gold** = curated **table** with **expectations** (e.g. drop rows that violate a rule).


### Where to create a pipeline in the UI

Labels vary by workspace version; look under **Workflows** for **Pipelines** or **Delta Live Tables**.

1. Sidebar → **Workflows** (sometimes grouped under data engineering).
2. Open **Pipelines** (or **Delta Live Tables** / **DLT**).
3. **Create pipeline**.
4. **Name:** e.g. `day07-dlt-{STUDENT_ID}`.
5. **Pipeline mode:** for class, **Triggered** is usually correct (see below).
6. **Libraries:** add **three** files (order does not matter — DLT builds the graph from `@dlt` definitions). Choose **one** approach:
   - **Notebooks:** `03-Day7-DLT-Bronze-Layer.ipynb`, `04-Day7-DLT-Silver-Layer.ipynb`, `05-Day7-DLT-Gold-Layer.ipynb`
   - **Python (Lakeflow / DLT .py modules):** `pipelines/dlt_bronze_flights.py`, `pipelines/dlt_silver_flights.py`, `pipelines/dlt_gold_flights.py`  
   For `.py`, set optional configuration **`bronze.source.delta.path`** if your source Delta path differs from the default in `dlt_bronze_flights.py`.
7. **Storage / catalog:** use what your instructor specifies (**Unity Catalog** vs legacy metastore).
8. **Target** (catalog + **schema**): a location your identity may write to, e.g. `main.day07_u25`. Managed tables appear there with names derived from your `@dlt.table` / `@dlt.view` function names unless you override with `name=`.
9. **Cluster / policy:** choose a policy that allows DLT pipelines; if creation fails, ask which policy to use.
10. **Start** (or **Refresh**) to run.

**After a run:** open the pipeline → **Lineage** (or graph): `bronze_flights` → `silver_flights` → `gold_flights`. Use **Events** / **Updates** for logs when debugging.


### Wiring multiple libraries into one pipeline

- Each **notebook** or **.py** module can define one or more `@dlt.table` / `@dlt.view` functions. All libraries in the pipeline contribute to a **single** dependency graph.
- Downstream code uses **`dlt.read("logical_name")`**, where `logical_name` is the **Python function name** of the upstream object (e.g. `bronze_flights`).
- Running library code on an **interactive** cluster is not the same as a pipeline run; `import dlt` may fail or behave differently. Always validate using **Start** on the pipeline.
- After you change transformation logic, use **Refresh**; use **Full refresh** when the instructor asks for a clean rebuild (slower, rewrites materialized tables).


### Triggered vs continuous

| Mode | Behaviour | When to use |
|------|-----------|-------------|
| **Triggered** | Runs on demand or on a schedule; cluster often stops between runs. | Class demos, batch medallion, scheduled loads. |
| **Continuous** | Long-running; pairs with `readStream` in live tables. | Near real-time streams; higher cost; often restricted. |

This bronze layer uses **batch** `spark.read` on Delta, so **Triggered** is the normal choice.

### SQL equivalence (reference)

Course-style expectation in SQL pipelines:

```sql
CONSTRAINT valid_count
EXPECT (count >= 0)
ON VIOLATION DROP ROW
```

Notebook **05** uses **`@dlt.expect_or_drop`**, which matches that **drop row** behaviour for the gold table.


### Troubleshooting

| Symptom | What to check |
|---------|----------------|
| `dlt` import or pipeline API errors when running a library notebook **interactively** | Run the pipeline from **Workflows → Pipelines** instead. |
| **Table or view not found** | All three libraries (notebooks **or** `.py` files) must be in the **same** pipeline; `dlt.read("bronze_flights")` must match the bronze **function name**. |
| **Cannot read P_BASIC** | Pipeline cluster identity needs **read** on the Day 5 ABFS path; re-run Day 5 notebook on the same workspace to confirm. |
| **Permission denied** on target schema | Grant **USE SCHEMA** / **CREATE TABLE** (or equivalent) for the pipeline identity on UC. |
| Run **stuck** or **queued** | Check **Events**; cancel duplicate runs if needed. |
| **All rows dropped** by expectations | Inspect bronze sample data; adjust `@dlt.expect_or_drop` for the exercise. |
| Stale results after code edits | Run **Full refresh** once. |

**Documentation:** [Delta Live Tables](https://docs.databricks.com/delta-live-tables/index.html), [Python reference](https://docs.databricks.com/delta-live-tables/python-ref.html), [Expectations](https://docs.databricks.com/delta-live-tables/expectations.html).


### Suggested order (item 20)

1. Prerequisite cell in **this** notebook (interactive cluster).
2. Create the pipeline with three libraries — notebooks **03**, **04**, **05** **or** `pipelines/dlt_bronze_flights.py`, `dlt_silver_flights.py`, `dlt_gold_flights.py` — then **Start**.
3. Verify lineage and query **`gold_flights`** from the target schema (SQL warehouse or notebook).

Optional streaming DLT (`readStream`) and extra labs: see `labs.md` and any instructor-provided bundle paths.
